In [1]:
%pip install dagshub

  Using cached dagshub-0.7.0-py3-none-any.whl.metadata (12 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached gitpython-3.1.50-py3-none-any.whl.metadata (14 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached dacite-1.6.0-py3-none-any.whl.metadata (14 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached gql-4.0.0-py3-none-any.whl.metadata (10 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached treelib-1.8.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pathvalidate-3.3.1-py3-none-any.whl.metadata (12 kB)
  Using cached semver-3.0.4-py3-none-any.whl.metadata (6.8 kB)
  Using cached dagshub_annotation_converter-0.2.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached lxml-6.1.1-cp311-cp311-win_amd64.whl.metadata (3.6 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using 


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install mlflow

  Using cached mlflow-3.14.0-py3-none-any.whl.metadata (49 kB)
  Using cached mlflow_skinny-3.14.0-py3-none-any.whl.metadata (50 kB)
Using cached mlflow-3.14.0-py3-none-any.whl (12.6 MB)
Using cached mlflow_skinny-3.14.0-py3-none-any.whl (3.5 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached mlflow-3.14.0-py3-none-any.whl.metadata (49 kB)
     ---------------------------------------- 0.0/50.5 kB ? eta -:--:--
     --------------- ---------------------- 20.5/50.5 kB 330.3 kB/s eta 0:00:01
     ------------------------------ ------- 41.0/50.5 kB 495.5 kB/s eta 0:00:01
     -------------------------------------- 50.5/50.5 kB 432.8 kB/s eta 0:00:00
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached pandas-2.3.3-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached pyarrow-24.0.0-cp311-cp311-win_amd64.whl.metadata (3.0 kB)
  Using cached skops-0.14.0-py3-none-any.whl.metadata (4.4 kB)
  Using cached waitress-3.0.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached cachetools-7.1.4-py3-none-any.whl.metadata (5.5 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
     ------------------

  You can safely remove it manually.

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [60]:
import sys
sys.path = [p for p in sys.path if "spark" not in p.lower()]

import dagshub
import mlflow
import mlflow.data
import mlflow.sklearn
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, f1_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt


In [61]:

dagshub.init(repo_owner='aditya0589', repo_name='youtube-sentiment-classifier', mlflow=True)
    
    

Initialized MLflow to track repo "aditya0589/youtube-sentiment-classifier"

Repository aditya0589/youtube-sentiment-classifier initialized!

In [51]:
df = pd.read_csv('../data/final_features.csv')

In [52]:
df.head()

,text,label
0,Plzz uploaded gaming video,neutral
1,thats actually fucking dope bro,neutral
2,Im glad youre proud because after focusing on ...,positive
3,just came from mine all day i could cry rn,negative
4,watch me 5 years from now on i will come back ...,positive


In [53]:
X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
mlflow.sklearn.autolog()

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

param_grid = {
    'tfidf__max_features': [200, 500, 1000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__stop_words': ['english', None],
    'clf__C': [0.1, 1.0, 10.0],
    'clf__class_weight': ['balanced']
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='f1_weighted',
    n_jobs=-1
)
# 
mlflow.set_experiment('yt-comment-classifier-experimenta')
with mlflow.start_run(run_name="grid_search_baseline_800_rows"):
    
    train_df = X_train.to_frame(name="text")
    train_df["label"] = y_train.values
    test_df = X_test.to_frame(name="text")
    test_df["label"] = y_test.values
    train_df.to_csv("train.csv", index=False)
    test_df.to_csv("test.csv", index=False)
    mlflow.log_artifact("train.csv")
    mlflow.log_artifact("test.csv")
    
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    
    print("--- GRID SEARCH COMPLETE ---")
    print(f"Best Parameters: {grid_search.best_params_}")
    print(f"Best Cross-Validation Score (Weighted F1): {grid_search.best_score_:.4f}")
    print("\nHoldout Test Set Performance:")
    print(classification_report(y_test, y_pred))

2026/06/30 17:37:57 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/06/30 17:38:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/30 17:38:01 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\kanak\AppData\Local\Temp\tmp1kz_1fzj\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.9.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/06/30 17:38:04 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
c

--- GRID SEARCH COMPLETE ---
Best Parameters: {'clf__C': 1.0, 'clf__class_weight': 'balanced', 'tfidf__max_features': 1000, 'tfidf__ngram_range': (1, 1), 'tfidf__stop_words': None}
Best Cross-Validation Score (Weighted F1): 0.6744

Holdout Test Set Performance:
              precision    recall  f1-score   support

    negative       0.54      0.54      0.54        24
     neutral       0.55      0.61      0.57        38
    positive       0.84      0.81      0.82        99

    accuracy                           0.72       161
   macro avg       0.64      0.65      0.65       161
weighted avg       0.73      0.72      0.72       161

🏃 View run grid_search_baseline_800_rows at: https://dagshub.com/aditya0589/youtube-sentiment-classifier.mlflow/#/experiments/3/runs/576923036d8c48e79f25ebeb8173423c
🧪 View experiment at: https://dagshub.com/aditya0589/youtube-sentiment-classifier.mlflow/#/experiments/3


In [ ]:
mlflow.sklearn.autolog()

svc_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LinearSVC(random_state=42, max_iter=2000))
])

svc_param_grid = {
    'tfidf__max_features': [500, 1000, 2000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'clf__C': [0.01, 0.1, 1.0, 10.0],
    'clf__class_weight': ['balanced']  # Keep class balancing active
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search_svc = GridSearchCV(
    estimator=svc_pipeline,
    param_grid=svc_param_grid,
    cv=cv,
    scoring='f1_weighted',
    n_jobs=-1
)

mlflow.set_experiment('yt-comment-classifier-experimenta')
with mlflow.start_run(run_name="grid_search_linearsvc"):
    grid_search_svc.fit(X_train, y_train)
    best_svc = grid_search_svc.best_estimator_
    y_pred_svc = best_svc.predict(X_test)
    
    print("--- LINEAR SVC GRID SEARCH COMPLETE ---")
    print(f"Best Parameters: {grid_search_svc.best_params_}")
    print(f"Best Cross-Validation Score (Weighted F1): {grid_search_svc.best_score_:.4f}")
    print("\nHoldout Test Set Performance:")
    print(classification_report(y_test, y_pred_svc))


2026/06/30 17:46:55 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2026/06/30 17:47:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/30 17:47:35 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\kanak\AppData\Local\Temp\tmpwgm5kllv\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.9.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/06/30 17:49:27 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.


🏃 View run grid_search_linearsvc at: https://dagshub.com/aditya0589/youtube-sentiment-classifier.mlflow/#/experiments/3/runs/7f0aa95c00ac49a886e761dc430575a6
🧪 View experiment at: https://dagshub.com/aditya0589/youtube-sentiment-classifier.mlflow/#/experiments/3


KeyboardInterrupt: 

In [ ]:
mlflow.sklearn.autolog(disable=True)

svc_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LinearSVC(random_state=42, max_iter=2000))
])
svc_param_grid = {
    'tfidf__max_features': [500, 1000, 2000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'clf__C': [0.1, 1.0, 10.0],
    'clf__class_weight': ['balanced']
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search_svc = GridSearchCV(
    estimator=svc_pipeline,
    param_grid=svc_param_grid,
    cv=cv,
    scoring='f1_weighted',
    n_jobs=1  
)

grid_search_svc.fit(X_train, y_train)
best_svc = grid_search_svc.best_estimator_
y_pred_svc = best_svc.predict(X_test)
accuracy = accuracy_score(y_test, y_pred_svc)
precision = precision_score(y_test, y_pred_svc, average='weighted')
recall = recall_score(y_test, y_pred_svc, average='weighted')
f1 = f1_score(y_test, y_pred_svc, average='weighted')
mlflow.set_experiment('yt-comment-classifier-experimenta')
with mlflow.start_run(run_name="manual_best_linearsvc"):
    mlflow.log_params(grid_search_svc.best_params_)
    
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    
    mlflow.log_artifact("train.csv")
    mlflow.log_artifact("test.csv")
    
    print("--- LINEAR SVC GRID SEARCH COMPLETE ---")
    print(f"Best Parameters: {grid_search_svc.best_params_}")
    print(f"Best Cross-Validation Score (Weighted F1): {grid_search_svc.best_score_:.4f}")
    print("\nHoldout Test Set Performance:")
    print(classification_report(y_test, y_pred_svc))

--- LINEAR SVC GRID SEARCH COMPLETE ---
Best Parameters: {'clf__C': 1.0, 'clf__class_weight': 'balanced', 'tfidf__max_features': 1000, 'tfidf__ngram_range': (1, 1)}
Best Cross-Validation Score (Weighted F1): 0.6836

Holdout Test Set Performance:
              precision    recall  f1-score   support

    negative       0.50      0.46      0.48        24
     neutral       0.58      0.55      0.57        38
    positive       0.81      0.84      0.82        99

    accuracy                           0.71       161
   macro avg       0.63      0.62      0.62       161
weighted avg       0.71      0.71      0.71       161

🏃 View run manual_best_linearsvc at: https://dagshub.com/aditya0589/youtube-sentiment-classifier.mlflow/#/experiments/3/runs/87d940989fd543278a66905bddeeb80b
🧪 View experiment at: https://dagshub.com/aditya0589/youtube-sentiment-classifier.mlflow/#/experiments/3


In [ ]:
mlflow.sklearn.autolog(disable=True)

nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', MultinomialNB())
])

nb_param_grid = {
    'tfidf__max_features': [500, 1000, 2000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__stop_words': ['english', None],
    'clf__alpha': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search_nb = GridSearchCV(
    estimator=nb_pipeline,
    param_grid=nb_param_grid,
    cv=cv,
    scoring='f1_weighted',
    n_jobs=1  
)

grid_search_nb.fit(X_train, y_train)

best_nb = grid_search_nb.best_estimator_
y_pred_nb = best_nb.predict(X_test)
accuracy = accuracy_score(y_test, y_pred_nb)
precision = precision_score(y_test, y_pred_nb, average='weighted')
recall = recall_score(y_test, y_pred_nb, average='weighted')
f1 = f1_score(y_test, y_pred_nb, average='weighted')
mlflow.set_experiment('yt-comment-classifier-experimenta')
with mlflow.start_run(run_name="manual_best_naivebayes"):

    mlflow.log_params(grid_search_nb.best_params_)
    

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    
 
    mlflow.log_artifact("train.csv")
    mlflow.log_artifact("test.csv")
    
    print("--- NAIVE BAYES GRID SEARCH COMPLETE ---")
    print(f"Best Parameters: {grid_search_nb.best_params_}")
    print(f"Best Cross-Validation Score (Weighted F1): {grid_search_nb.best_score_:.4f}")
    print("\nHoldout Test Set Performance:")
    print(classification_report(y_test, y_pred_nb))

--- NAIVE BAYES GRID SEARCH COMPLETE ---
Best Parameters: {'clf__alpha': 0.01, 'tfidf__max_features': 2000, 'tfidf__ngram_range': (1, 1), 'tfidf__stop_words': 'english'}
Best Cross-Validation Score (Weighted F1): 0.6224

Holdout Test Set Performance:
              precision    recall  f1-score   support

    negative       0.54      0.29      0.38        24
     neutral       0.56      0.26      0.36        38
    positive       0.68      0.89      0.77        99

    accuracy                           0.65       161
   macro avg       0.59      0.48      0.50       161
weighted avg       0.63      0.65      0.61       161

🏃 View run manual_best_naivebayes at: https://dagshub.com/aditya0589/youtube-sentiment-classifier.mlflow/#/experiments/3/runs/9b58595d7b794cc081eba4d20d5de92f
🧪 View experiment at: https://dagshub.com/aditya0589/youtube-sentiment-classifier.mlflow/#/experiments/3
